In [ ]:
import sys
import os
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, SubsetRandomSampler
from tqdm import tqdm
import argparse
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

cwd = Path.cwd()
print("Current working directory:", cwd) # /path/to/home
root_path = Path("/path/to/BrainWear_Kareem")
# root_path = Path("/path/to/BrainWear_Kareem")
project_root = root_path/"FYP"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Added to path: {project_root}")

from datasets.brainwear import BrainWearDataset
from baseline.autoencoder.models import ResNetAutoencoder, ResNetClassifier

DEFAULT_DATA_DIR = root_path/"Processed_Brainwear_new"
MODEL_DIR = project_root/"baseline"/"autoencoder"/"models"

In [ ]:
# Gradual unfreezing helpers

# We unfreeze top-to-bottom so the task-relevant features adapt first.
ENCODER_LAYER_GROUPS = ["layer4", "layer3", "layer2", "layer1", "conv1"]

def get_unfreezing_schedule(total_epochs: int, num_groups: int = len(ENCODER_LAYER_GROUPS)) -> dict[int, str]:
    """
    Returns a dict mapping {epoch_number: layer_group_name_to_unfreeze}
    """
    phase_len = total_epochs // (num_groups + 1)
    schedule = {}
    for i, group in enumerate(ENCODER_LAYER_GROUPS):
        trigger_epoch = phase_len * (i + 1)   # 0-indexed epoch
        schedule[trigger_epoch] = group
    return schedule


def unfreeze_layer_group(model: nn.Module, group_name: str):
    """
    Encoder lives at model.encoder.feature_extractor (an nn.Sequential)
    inside ResNetSpatialEncoder.
    """
    unfrozen_count = 0
    for name, param in model.encoder.named_parameters():
        if group_name in name:
            param.requires_grad = True
            unfrozen_count += 1
    print(f"  [Unfreeze] '{group_name}': {unfrozen_count} parameters unfrozen.")


def build_optimizer(model: nn.Module, base_lr: float, weight_decay: float,
                    decay_factor: float) -> optim.AdamW:
    """
    Build AdamW optimiser with discriminative learning rates:
      - Classification head  -> base_lr
      - encoder.projection   -> base_lr * decay_factor
      - layer4               -> base_lr * decay_factor^2
      - layer3               -> base_lr * decay_factor^3
      - layer2               -> base_lr * decay_factor^4
      - layer1               -> base_lr * decay_factor^5
      - conv1/bn1 (stem)     -> base_lr * decay_factor^6
    """
    # Collect named parameter groups in priority order
    group_defs = [
        ("classifier",          base_lr),
        ("pool",                base_lr * decay_factor),
        ("encoder.layer4",      base_lr * (decay_factor ** 2)),
        ("encoder.layer3",      base_lr * (decay_factor ** 3)),
        ("encoder.layer2",      base_lr * (decay_factor ** 4)),
        ("encoder.layer1",      base_lr * (decay_factor ** 5)),
        ("encoder.conv1",       base_lr * (decay_factor ** 6)),
        ("encoder.bn1",         base_lr * (decay_factor ** 6)),
    ]

    # Assign each trainable param to the first matching group
    assigned = set()
    param_groups = []
    for group_key, lr in group_defs:
        params = [
            p for name, p in model.named_parameters()
            if group_key in name and p.requires_grad and id(p) not in assigned
        ]
        if params:
            for p in params:
                assigned.add(id(p))
            param_groups.append({"params": params, "lr": lr})
            print(f"[Optimizer] group '{group_key}': {len(params)} tensors, lr={lr:.2e}")

    # Catch any trainable params not matched above (safety net)
    leftover = [p for p in model.parameters() if p.requires_grad and id(p) not in assigned]
    if leftover:
        param_groups.append({"params": leftover, "lr": base_lr})
        print(f"[Optimizer] group 'leftover': {len(leftover)} tensors, lr={base_lr:.2e}")

    return optim.AdamW(param_groups, weight_decay=weight_decay)

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", type=str, default=str(DEFAULT_DATA_DIR))
parser.add_argument("--ae_weights", type=str, required=True, help="Path to saved autoencoder .pt file")
parser.add_argument("--batch_size", type=int, default=4)
parser.add_argument("--epochs", type=int, default=50)
parser.add_argument("--lr", type=float, default=5e-4)
parser.add_argument("--dropout", type=float, default=0.3)
parser.add_argument("--weight_decay", type=float, default=1e-4)
parser.add_argument("--num_classes", type=int, default=5, help="Number of Brainwear classes")
parser.add_argument("--folds", type=int, default=5, help="Number of cross-validation folds")
parser.add_argument("--quantile", action="store_true", help="Whether to use quantile-based binning for scores")
parser.add_argument("--patience", type=int, default=10)
parser.add_argument("--max_patients", type=int, default=None)
parser.add_argument("--score_file", type=str, default=str(root_path / "eortc_scores.csv"))
# Gradual unfreezing: LR multiplier applied to each successive encoder layer group
parser.add_argument("--unfreeze_lr_decay", type=float, default=0.1,
                    help="LR decay factor per encoder layer group (discriminative LRs)")

ae_weights_name = "resnet50_150e_0.0004lr.pt"
model_name = ae_weights_name.split("_")[0]

args = parser.parse_args([
    '--ae_weights', str(MODEL_DIR/"encoder"/ae_weights_name),
    '--epochs', '40',
    '--patience', '20',
    '--num_classes', '3',
    '--folds', '5',
    '--dropout', '0.6',
    # '--quantile'
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dir_name = f"{args.epochs}e_{args.lr}lr_{args.folds}k_{args.num_classes}c_cv_gradual_unfreeze_weighted_cost"
dir_name += "_quantile" if args.quantile else ""
run_dir = MODEL_DIR / "classifier" / dir_name
os.makedirs(run_dir, exist_ok=True)
print(f"Created run directory: {run_dir}")


print("Initialising Brainwear Dataset")
full_dataset = BrainWearDataset(
    root_dir=args.data_dir,
    target_size=(64, 96, 96),
    max_patients=args.max_patients,
    score_file=args.score_file,
    num_bins=args.num_classes,
    quantile_bins=args.quantile
)
total_size = len(full_dataset)

print("Loading pre-trained autoencoder features")
autoencoder = ResNetAutoencoder(model_name=model_name)
autoencoder.load_state_dict(torch.load(args.ae_weights, map_location=device), strict=False)

labels = [full_dataset[i][1] for i in range(len(full_dataset))]
kfold  = StratifiedKFold(n_splits=args.folds, shuffle=True, random_state=42)
cv_results = {}

# Pre-compute the unfreezing schedule once (same for every fold)
# unfreeze_schedule = get_unfreezing_schedule(args.epochs)
# print(f"Unfreezing schedule (epoch -> group): {unfreeze_schedule}")

# Trigger unfreezing at 80% of the way through training
unfreeze_epoch = int(args.epochs * 0.8)
print(f"Neutered Unfreezing: Layer 4 will unfreeze at epoch {unfreeze_epoch}")

print(f"Starting {args.folds}-Fold Cross Validation Training")


# CV loop
for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(total_size), labels)):
    from collections import Counter
    train_dist = Counter(labels[i] for i in train_idx)
    val_dist   = Counter(labels[i] for i in val_idx)
    print(f"  Fold {fold+1} train distribution: {dict(sorted(train_dist.items()))}")
    print(f"  Fold {fold+1} val   distribution: {dict(sorted(val_dist.items()))}")
    print(f"\n================ Starting Fold {fold + 1}/{args.folds} ================")

    # Compute class weights from this fold's training labels to handle class imbalance
    train_labels_arr = np.array([labels[i] for i in train_idx])
    raw_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(args.num_classes),
        y=train_labels_arr,
    )
    class_weights_tensor = torch.tensor(raw_weights, dtype=torch.float).to(device)
    classification_loss = nn.CrossEntropyLoss(weight=class_weights_tensor)
    print(f"  Fold {fold+1} class weights: { {c: round(w, 3) for c, w in enumerate(raw_weights)} }")

    train_sampler = SubsetRandomSampler(train_idx)
    val_sampler   = SubsetRandomSampler(val_idx)

    train_loader = DataLoader(full_dataset, batch_size=args.batch_size,
                              sampler=train_sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(full_dataset, batch_size=args.batch_size,
                              sampler=val_sampler,   num_workers=4, pin_memory=True)

    # Encoder starts fully frozen for this fold
    model = ResNetClassifier(
        autoencoder=autoencoder,
        num_classes=args.num_classes,
        dropout=args.dropout,
        unfreeze_encoder=False,
    ).to(device)

    model.set_encoder_frozen(True)

    # Initial optimizer - ONLY the classification head is trainable
    print("[Optimizer] Initial setup (head only):")
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.AdamW(trainable_params, lr=args.lr, weight_decay=args.weight_decay)
    # optimizer = build_optimizer(model, args.lr, args.weight_decay, args.unfreeze_lr_decay)        # Gradual unfreezing
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss    = float('inf')
    best_val_acc     = 0.0
    best_epoch       = 0
    epochs_no_improve = 0
    save_location    = run_dir / f"best_fold_{fold + 1}.pt"

    for epoch in range(args.epochs):

        # Gradual unfreezing trigger
        # if epoch in unfreeze_schedule:
        #     group_to_unfreeze = unfreeze_schedule[epoch]
        #     print(f"\n  [Epoch {epoch + 1}] Unfreezing encoder group: '{group_to_unfreeze}'")
        #     unfreeze_layer_group(model, group_to_unfreeze)

        #     # Rebuild optimiser so newly unfrozen params get their own
        #     # momentum state and the correct discriminative LR.
        #     print(f"  [Epoch {epoch + 1}] Rebuilding optimiser with discriminative LRs:")
        #     optimizer = build_optimizer(model, args.lr, args.weight_decay, args.unfreeze_lr_decay)
        #     # Re-create scheduler referencing the new optimiser
        #     scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        #         optimizer, mode='min', factor=0.5, patience=5
        #     )
        #     epochs_no_improve = 0
        #     print(f"  [Unfreeze] Patience counter reset.")

        if epoch == unfreeze_epoch:
            print(f"\n  [Epoch {epoch + 1}] Unfreezing only ResNet layer4")
            
            # Target only layer4 in the encoder to be unfrozen
            for name, param in model.encoder.named_parameters():
                if "layer4" in name:
                    param.requires_grad = True
                else:
                    param.requires_grad = False # Keep conv1, layer1, layer2, layer3 locked
            
            model._encoder_frozen = False 

            # Rebuild optimizer with Discriminative Learning Rates
            print(f"  [Epoch {epoch + 1}] Rebuilding optimizer with discriminative LRs:")
            
            layer4_params = [p for n, p in model.encoder.named_parameters() if "layer4" in n and p.requires_grad]
            head_params = [p for p in model.classifier.parameters() if p.requires_grad]

            optimizer = optim.AdamW([
                {'params': layer4_params, 'lr': args.lr * 0.1}, # 10x smaller LR for the pre-trained layer4
                {'params': head_params, 'lr': args.lr}          # Standard LR for the classification head
            ], weight_decay=args.weight_decay)
            
            # Re-create scheduler referencing the new optimizer
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=5
            )
            epochs_no_improve = 0
            print(f"  [Unfreeze] Patience counter reset.")

        # Train
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        progress_bar = tqdm(
            train_loader,
            desc=f"Fold {fold+1} Epoch {epoch+1}/{args.epochs} [Train]",
            leave=False,
        )

        for t2_imgs, batch_labels in progress_bar:
            t2_imgs, batch_labels = t2_imgs.to(device), batch_labels.to(device)

            optimizer.zero_grad()
            logits = model(t2_imgs)
            loss   = classification_loss(logits, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            train_total   += batch_labels.size(0)
            train_correct += (predicted == batch_labels).sum().item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

        avg_train_loss = train_loss / len(train_loader)
        train_acc      = 100 * train_correct / train_total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for t2_imgs, batch_labels in val_loader:
                t2_imgs, batch_labels = t2_imgs.to(device), batch_labels.to(device)
                logits = model(t2_imgs)
                loss   = classification_loss(logits, batch_labels)
                val_loss    += loss.item()
                _, predicted = torch.max(logits.data, 1)
                val_total   += batch_labels.size(0)
                val_correct += (predicted == batch_labels).sum().item()

        avg_val_loss = val_loss / len(val_loader)
        val_acc      = 100 * val_correct / val_total

        scheduler.step(avg_val_loss)

        # Log current LR of the head group as a proxy
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"F{fold+1} Epoch [{epoch+1}/{args.epochs}] | "
            f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.2f}% | "
            f"LR (head): {current_lr:.2e}"
        )

        # Checkpoint logic
        if avg_val_loss < best_val_loss:
            best_val_loss    = avg_val_loss
            best_val_acc     = val_acc
            best_epoch       = epoch + 1
            epochs_no_improve = 0
            print(f"New best at epoch {best_epoch}. Val Loss: {best_val_loss:.4f}. Acc: {best_val_acc:.2f}%")
            torch.save(model.state_dict(), save_location)
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s)")
            if epochs_no_improve >= args.patience:
                print(f"Early stopping triggered. Best Val Loss: {best_val_loss:.4f}")
                break

    cv_results[f"fold_{fold+1}"] = {
        "best_epoch":      best_epoch,
        "best_val_loss":   best_val_loss,
        "best_val_acc":    best_val_acc,
        "final_train_acc": train_acc,
    }
    print(f"Fold {fold+1} Complete. Best Val Acc: {best_val_acc:.2f}%")


# Aggregate results
print("\n============ Cross-Validation Complete ============")
avg_cv_loss = sum(f["best_val_loss"] for f in cv_results.values()) / args.folds
avg_cv_acc  = sum(f["best_val_acc"]  for f in cv_results.values()) / args.folds

print(f"Average CV Validation Accuracy: {avg_cv_acc:.2f}%")
print(f"Average CV Validation Loss:     {avg_cv_loss:.4f}")

run_details = {
    "arguments": vars(args),
    # "unfreeze_schedule": {str(k): v for k, v in unfreeze_schedule.items()},
    "cross_validation_metrics": {
        "average_val_loss": avg_cv_loss,
        "average_val_acc":  avg_cv_acc,
    },
    "fold_details": cv_results,
}

json_path = run_dir / "cv_run_details.json"
with open(json_path, "w") as json_file:
    json.dump(run_details, json_file, indent=4)

print(f"Details saved to {json_path}")